# First Notebook

This first activity will use basic python to load and visualize data. We will:

- Load three datasets — **Bahamian GDP**, **Bahamian nighttime lights**, and the **S&P 500**
- Perform any necessary data cleaning
- Reshape/Resample/Merge as needed
- Calculate basic descriptives
- Plot the data

The data is available in the ```data``` folder.

*Make sure the folder is in the same directory as this notebook.*

This Notebook has **parts** of the code you need to to the tasks above. Follow the notes to complete or write the pieces of code you may need to finish.



In [ ]:
# Verify this line runs correctly.
# Otherwise, go to act0_env_check.ipynb

import pandas as pd  # import the pandas library and give it the alias "pd"

Load the GDP data

In [ ]:
df_gdp = pd.read_csv('../data/gdp_bs.csv',   # Bahamian real GDP; path is relative to this notebook
                     index_col='date',   # use the 'date' column as the row index instead of 0, 1, 2, ...
                     parse_dates=True)   # convert the index to datetime objects automatically
df_gdp = df_gdp.dropna() # This line drops missing values

df_gdp.rename(columns={'RGDP0000': 'gdp'}, inplace=True)  # rename the column to something shorter
# Observe the data
df_gdp.tail(5)  # show the last 5 rows of the DataFrame

It has a single series, `gdp` (quarterly, in millions). Select it explicitly — the
double brackets keep the result a **DataFrame** rather than a Series.

In [ ]:
df_gdp = df_gdp[['gdp']]
df_gdp.head(5)  # show the first 5 rows of the DataFrame

**COMPLETE.**

Load the stock data.

In [ ]:
# YOUR CODE HERE: call pd.`read_csv(...) with the stock file path, index_col='date', parse_dates=True\
df_stock = pd.read_csv( <YOUR CODE HERE: load sp500.csv> , index_col='date', parse_dates=True)

df_stock.tail(5)  # show the last 5 rows to verify the data loaded correctly

Open the nighttime lights data: `data/blk_ntl.csv`, NASA's **monthly**
cloud-free composite of artificial light over The Bahamas.

In [ ]:
df_ntl = <YOUR CODE HERE: load blk_ntl.csv>  # load the nightlights data into a DataFrame

df_ntl.head(10)  # show the first 10 rows (.tail shows last; .head shows first)

The file has two columns. Keep only `NearNadir_Composite_Snow_Free` — the monthly
composite. The other column is a placeholder and is empty for all but one month.

In [ ]:
# Double brackets [[ ]] select a subset of columns and return a DataFrame (single brackets return a Series)
# YOUR CODE HERE: put the column name you want to keep inside the inner brackets,
# e.g. df_ntl[['some_column']]   (the text above tells you which one)
df_ntl = <YOUR CODE HERE: select the relevant column>
df_ntl.head(5)

Let's start by visualizing

In [ ]:
<YOUR CODE HERE: use the .plot() function to plot NTL data>

Notice the three series are on **three different frequencies**: the lights are
monthly, GDP is quarterly, and the stock index is daily. Series cannot be merged
until they share a frequency, and the slowest one — GDP — decides it.

Resample the lights down to quarterly.

**COMPLETE**: **QS** standds for Quarter Start. **QE** for Quarter End.

In [ ]:
# YOUR CODE HERE: pass a frequency string inside resample(), e.g. resample('QS') for Quarter Start
# .mean() then averages all monthly values that fall within each quarter
df_ntl = df_ntl.<YOUR CODE HERE: call .resample(format)>.mean()
df_ntl.tail(10)

You will notice your data has many missing values because GDP finishes in 2023 but the stock index runs to 2026. Let's keep only the columns where there are no missing values.

In [ ]:
# dropna removes rows with missing values (NaN)
# how='all' means: only drop a row if ALL its columns are NaN (keeps rows missing just one value)
# axis=0 means operate on rows (axis=1 would operate on columns)
df_ntl = df_ntl.dropna(how='all', axis=0)
df_ntl.tail(10)

Descriptive statistics here

In [ ]:
df_ntl.describe()  # returns count, mean, std, min, quartiles, and max for each numeric column

**COMPLETE**: Can we plot again?

In [ ]:
<YOUR CODE HERE: plot df_ntl>

# Merge data

We have 3 series

In [ ]:
# Calling .plot() on each DataFrame opens a separate chart for each series
# Expressions separated by commas inside a cell return a tuple of results — each plot displays inline
df_ntl.plot(), df_gdp.plot(), df_stock.plot()

Resample stock data into quarters

In [ ]:
df_stock = <YOUR CODE HERE: resample> # group daily/monthly data into quarters (Quarter Start) and average

df_stock.tail(10)

Now merge, but...

In [ ]:
# .merge() joins two DataFrames — like a SQL JOIN
# left_index=True, right_index=True means use the row index (date) as the join key on both sides
df_merge = df_ntl.merge(df_gdp, left_index=True, right_index=True)
df_merge = df_merge.merge(df_stock, left_index=True, right_index=True)  # chain a second merge onto the result
df_merge

It's not working. This is a common problem. The date indexes do not align. Look at GDP again:

In [ ]:
df_gdp.tail(5)

In [ ]:
df_stock.tail(5)

In [ ]:
df_ntl.tail(5)

GDP is stamped at the **start of the last month** of each quarter (1 Mar, 1, Jun)
while `resample('QS')` gave the other two the **start** (1 Jan, 1 Apr, …). Several fixes exist — you could
resample GDP to `QS` or move the other dates to the last month of the quarter.

In [ ]:
df_gdp = <YOUR CODE HERE: resample GDP>

In [ ]:
# Now that the date indexes match, the merge should produce non-empty rows
df_merge = df_ntl.merge(df_gdp, left_index=True, right_index=True)
df_merge = df_merge.merge(df_stock, left_index=True, right_index=True)
df_merge

Plot

In [ ]:
df_merge.plot()

Yes, but we may want to make the comparison meaningful. Let's normalize both series to the QoQ rate of change.

In [ ]:
df_change = df_merge.pct_change()  # compute period-over-period percentage change: (current - previous) / previous
df_change.head(5)  # first row will be NaN because there is no previous value to compare against

Plot

In [ ]:
df_change.plot()